# 04 — Explicabilité et équité du modèle

**Projet :** Cert_EET — prédiction précoce des apprenants à risque
**Auteur :** Iyed Omri — PFE ESPRIT 2026

Un formateur ne peut pas agir sur une simple probabilité. Il doit savoir **pourquoi** un apprenant
est signalé. Dans ce notebook :
1. j'explique le modèle **globalement** (quelles variables comptent le plus) ;
2. j'explique **chaque prédiction** avec ses trois principales raisons, en français simple ;
3. je vérifie que le modèle fonctionne **aussi bien pour tous les groupes** (genre, âge, handicap).

**Méthode : SHAP.** Pour chaque apprenant, SHAP répartit la prédiction entre les variables : chaque
variable reçoit une **contribution** positive (elle augmente le risque) ou négative (elle le diminue).
LightGBM calcule ces valeurs lui-même (algorithme TreeSHAP), sans bibliothèque supplémentaire.m

In [ ]:
import json
import sys
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import precision_score, recall_score, roc_auc_score

RACINE_ML = next(p for p in [Path.cwd(), Path.cwd().parent] if (p / "risque").exists())
sys.path.insert(0, str(RACINE_ML))
from risque.explications import niveau, raisons

modele = joblib.load(RACINE_ML / "modeles" / "risque_v1.joblib")
fiche = json.loads((RACINE_ML / "modeles" / "risque_v1.json").read_text(encoding="utf-8"))
ENTREES, SEUIL = fiche["entrees"], fiche["seuil"]

df = pd.read_csv(RACINE_ML / "donnees" / "variables.csv.gz")
test = df[df["code_presentation"] == "2014J"].reset_index(drop=True)   # même test que le notebook 03
X_test = test[ENTREES]

proba = modele.predict_proba(X_test)[:, 1]
contrib = modele.predict(X_test, pred_contrib=True)   # une colonne par variable + la valeur de base
shap_valeurs = pd.DataFrame(contrib[:, :-1], columns=ENTREES)

# Vérification : base + somme des contributions = score brut du modèle
ecart = np.abs(contrib.sum(axis=1) - modele.predict(X_test, raw_score=True)).max()
print(f"Modèle : {fiche['modele']} | seuil : {SEUIL} | écart maximal de la vérification SHAP : {ecart:.2e}")

## 1. Explication globale : quelles variables comptent le plus ?

Pour chaque variable, je calcule la **moyenne de la valeur absolue** de sa contribution sur tous les
apprenants du test. Plus la barre est longue, plus la variable influence les prédictions, dans un
sens ou dans l'autre.

In [ ]:
importance = shap_valeurs.abs().mean().sort_values()
ax = importance.plot.barh(figsize=(8, 6), color="#1565c0")
ax.set_title("Importance globale des variables (moyenne |SHAP|)")
ax.set_xlabel("Contribution moyenne (en log-odds)")
plt.tight_layout()
plt.show()
print(importance.sort_values(ascending=False).round(3))

## 2. Explication individuelle : les raisons de chaque alerte

Pour un apprenant, je garde les **trois contributions positives les plus fortes** (ce qui augmente
son risque) et je les traduis en phrases simples (`ml/risque/explications.py`).

Je n'affiche pas de pourcentage exact mais un **niveau** :
- **faible** : probabilité sous le seuil d'alerte du modèle ;
- **moyen** : au-dessus du seuil d'alerte ;
- **élevé** : 0,60 ou plus.

J'ai fait ce choix car la calibration (notebook 03) n'est plus fiable au-delà d'environ 0,65.

Exemples à J+28 : un apprenant à risque élevé, un à risque moyen, un à risque faible.

In [ ]:
test["proba"] = proba
test["niveau"] = [niveau(p, SEUIL) for p in proba]

j28 = test[test["coupure"] == 28]
for niv in ["élevé", "moyen", "faible"]:
    i = j28[j28["niveau"] == niv].sample(1, random_state=42).index[0]
    r = raisons(test.loc[i, ENTREES].to_dict(), shap_valeurs.loc[i].to_dict())
    print(f"Niveau {niv.upper()} (probabilité {proba[i]:.2f}) — résultat réel : {test.loc[i, 'final_result']}")
    for phrase in r or ["Aucun signal de risque particulier"]:
        print("   •", phrase)
    print()

print("Répartition des niveaux à J+28 :")
print(j28["niveau"].value_counts(normalize=True).mul(100).round(1))

## 3. Audit d'équité

Le genre, l'âge et le handicap **ne sont pas des entrées du modèle**. Je les utilise seulement ici,
pour vérifier que le modèle est aussi bon pour tous les groupes.

Comme les taux de risque réels diffèrent entre les groupes (notebook 01), je ne compare pas seulement
le nombre d'alertes. Je compare surtout :
- le **rappel** : un apprenant à risque a-t-il la même chance d'être repéré dans chaque groupe ?
- la **précision** : une alerte est-elle aussi fiable dans chaque groupe ?

Je compare aussi avec le rappel des **règles actuelles** : le modèle réduit-il ou augmente-t-il les
écarts ? L'âge est regroupé en deux classes (0–35 ans / 35 ans et plus), car le groupe des plus de
55 ans est trop petit. Le calcul porte sur toutes les dates de coupure de la session de test.

In [ ]:
def regles(t):   # même définition que dans le notebook 03
    return ((t["jours_depuis_derniere_activite"] > 7) | (t["echecs_consecutifs_fin"] >= 2)
            | (t["nb_non_rendues"] >= 1) | ((t["tendance_score"] <= -20) & (t["nb_notes"] >= 3))).astype(int)

test["alerte"] = (test["proba"] >= SEUIL).astype(int)
test["alerte_regles"] = regles(test)
test["age"] = np.where(test["age_band"] == "0-35", "0-35 ans", "35 ans et plus")
test["handicap"] = test["disability"].map({"N": "Non", "Y": "Oui"})
test["genre"] = test["gender"].map({"F": "Femme", "M": "Homme"})

lignes = []
for col in ["genre", "age", "handicap"]:
    for groupe, t in test.groupby(col):
        lignes.append({
            "variable": col, "groupe": groupe, "effectif": len(t),
            "taux_risque": t["a_risque"].mean(), "taux_alerte": t["alerte"].mean(),
            "rappel": recall_score(t["a_risque"], t["alerte"]),
            "precision": precision_score(t["a_risque"], t["alerte"]),
            "roc_auc": roc_auc_score(t["a_risque"], t["proba"]),
            "rappel_regles": recall_score(t["a_risque"], t["alerte_regles"]),
        })
equite = pd.DataFrame(lignes)
print(equite.round(3).to_string(index=False))

print("\nÉcart maximal de rappel entre groupes :")
for col, t in equite.groupby("variable"):
    print(f"  {col:9} modèle : {(t['rappel'].max() - t['rappel'].min())*100:.1f} points | "
          f"règles : {(t['rappel_regles'].max() - t['rappel_regles'].min())*100:.1f} points")

## 4. Mes constats

**1. Les explications sont exactes.** La somme des contributions SHAP redonne le score du modèle à
10⁻¹⁴ près : les raisons affichées décrivent vraiment ce que fait le modèle.

**2. Ce qui compte le plus pour le modèle :**
- les **notes** : `score_moyen` est de loin la variable la plus influente (0,64). Elle porte aussi
  l'information « pas encore de note », car LightGBM traite la valeur vide comme un cas à part.
  C'est pourquoi `a_au_moins_une_note` n'apporte plus rien ;
- les **remises** : `taux_remise` (0,34) ;
- l'**activité** : les quatre variables d'activité réunies pèsent environ 0,48, autant que les notes.

Les variables liées aux règles `ECHECS_QCM` et `SCORE_EN_CHUTE` (`echecs_consecutifs_fin`,
`tendance_score`) comptent très peu. Cela confirme le notebook 03 : tôt dans la session, il y a trop
peu de notes pour que ces signaux soient utiles.

**3. Les raisons sont compréhensibles par un formateur.** Exemple d'un apprenant à risque élevé
(probabilité 0,92, il a abandonné) : *aucun travail échu n'a été rendu ; aucune activité depuis
12 jours ; seulement 1 jour actif sur les 14 derniers jours.*

**4. Le modèle se trompe, et c'est pour cela qu'il ne décide pas seul.** Dans mes exemples, un
apprenant classé « moyen » a finalement réussi : il était inactif depuis 26 jours, une raison qui
justifie quand même un contact. Un apprenant classé « faible » a abandonné. Les raisons aident le
formateur à juger chaque cas ; la décision reste la sienne.

**5. Répartition des niveaux à J+28 :** faible 43 %, moyen 30 %, élevé 27 %. Le niveau **élevé** permet
au formateur de **prioriser** ses actions quand il a beaucoup d'apprenants signalés.

**6. Équité : le rappel est le même pour tous les groupes.**

| Groupe | Rappel du modèle | Rappel des règles |
|---|---|---|
| Femme / Homme | 0,80 / 0,78 | 0,45 / 0,42 |
| 0–35 ans / 35 ans et plus | 0,80 / 0,76 | 0,44 / 0,42 |
| Sans / avec handicap | 0,79 / 0,79 | 0,43 / 0,45 |

Les écarts de rappel restent sous 5 points (genre 1,4 ; âge 4,3 ; handicap 0,3). Un apprenant à
risque a donc presque la même chance d'être repéré, quel que soit son groupe. Et dans tous les groupes,
le modèle repère presque **deux fois plus** d'apprenants à risque que les règles.

**7. Mais un point d'attention : les femmes reçoivent plus de fausses alertes.** Pour les femmes, la
précision est de 0,47 contre 0,59 pour les hommes, et la ROC-AUC de 0,67 contre 0,77 : le modèle
distingue moins bien les femmes à risque des autres. Concrètement, 65 % des femmes reçoivent une
alerte alors que seulement 38 % sont réellement à risque. Une explication possible : les femmes et
les hommes ne suivent pas les mêmes modules dans OULAD, avec des rythmes d'activité différents.

**Ce que je décide :**
- je **ne corrige pas** cet écart avec un seuil différent par genre : il faudrait utiliser le genre
  comme entrée, ce que j'ai exclu, et Cert_EET ne collecte pas cette donnée ;
- je **documente** cette limite dans la fiche du modèle ;
- le formateur reste le décideur : une fausse alerte coûte un message, pas une sanction ;
- je refais cet audit lors du réentraînement sur les données de Cert_EET.